# 06 — Refund Prevention (Seller OS)

Refunds are won in the **first 2 hours**. This notebook gives a **complaint taxonomy**, **2h-SLA math**, the **money-at-risk formula (3 x $34.25 = $102.75)**, **4 before/after replies (EN)**, an **i18n pointer**, and a **refund/reship/escalate decision tree**. Stdlib arithmetic only.

## 0. Complaint language taxonomy

| bucket | signals | risk |
|---|---|---|
| late | `hasn't arrived`, `where is my`, `no tracking`, `late` | high |
| damaged | `chipped`, `damaged`, `broken`, `wrong` | high |
| refund-demand | `refund`, `return`, `chargeback`, `missing` | critical |
| question | `do you have`, `in blue`, `address change` | low |
| happy | `amazing`, `love`, `perfect`, `beautiful` | none (ask review) |

In [1]:
buckets = {'late': 3, 'damaged': 1, 'refund-demand': 2, 'question': 2, 'happy': 2}
total = sum(buckets.values())
for k, v in buckets.items():
    print(f"{k:13} {v} msgs ({v/total*100:.0f}%)")
print(f"at-risk = late+damaged+refund = {buckets['late']+buckets['damaged']+buckets['refund-demand']}")

late          3 msgs (30%)
damaged       1 msgs (10%)
refund-demand 2 msgs (20%)
question      2 msgs (20%)
happy         2 msgs (20%)
at-risk = late+damaged+refund = 6


### Keyword spotter (stdlib, word-boundary for `bad`/`late`)

Short words need `\\b` guards so `wherever` != `where` + `order`. Phrases (`hasn't arrived`, `no tracking`) match by substring.

In [2]:
import re
def is_negative(txt):
    low = txt.lower()
    if any(ph in low for ph in ["hasn't arrived", 'no tracking', 'where is my', 'never arrived']):
        return True
    return bool(re.search(r'\b(refund|damaged|late|broken|missing)\b', low))

tests = ['Hi! My tote has not arrived'.replace('has not', "hasn't"), 'Amazing mug! Love it!', 'Where is my order? No tracking.']
for t in tests:
    print(f"{is_negative(t)} <- {t[:40]}")

True <- Hi! My tote hasn't arrived
False <- Amazing mug! Love it!
True <- Where is my order? No tracking.


## 1. The 2-hour SLA — math, not vibes

`deadline = msg_time + 2h`. With 3 angry msgs and 1 human, budget **40 min each** (3 x 40m = 2h). Miss it → refund odds ~double (house rule from refundstorm shops).

In [3]:
msgs, sla_h, humans = 3, 2, 1
mins_each = sla_h*60*humans/msgs
print(f"{msgs} msgs / {humans} human / {sla_h}h SLA -> {mins_each:.0f} min each")
print(f"queue clears in {msgs*20} min at 20 min/reply (inside SLA)" if msgs*20 <= sla_h*60 else 'needs help')

3 msgs / 1 human / 2h SLA -> 40 min each
queue clears in 60 min at 20 min/reply (inside SLA)


### SLA clock — reply queue order

P0 = refund-demand first, then late/damaged, then questions. Happy msgs never block P0.

In [4]:
queue = [('refund-demand','P0'), ('late','P0'), ('damaged','P0'), ('question','P1'), ('happy','P2')]
for kind, pri in queue:
    print(f"{pri} {kind}")
print('rule: all P0 before P1')

P0 refund-demand
P0 late
P0 damaged
P1 question
P2 happy
rule: all P0 before P1


## 2. Money at risk — `queue x AOV`

Demo: 3 angry msgs x AOV $34.25 = **$102.75**. That is the headline number in `build_daily_close`.

In [5]:
aov, n = 34.25, 3
print(f"money_at_risk = {n} x ${aov:.2f} = ${n*aov:.2f}")
print(f"share of GMV ${274.00:.2f} = {n*aov/274.00*100:.1f}%")

money_at_risk = 3 x $34.25 = $102.75
share of GMV $274.00 = 37.5%


### What $102.75 means

37.5% of demo GMV sits in 3 threads. One batch reply session (<1h) protects it. Compare: ignoring it funds ~6 refund ships.

In [6]:
gmv, risk = 274.00, 102.75
print(f"at-risk share {risk/gmv*100:.1f}%")
print(f"profit ${169.09:.2f} - risk ${risk:.2f} = ${169.09-risk:.2f} if all refund")

at-risk share 37.5%
profit $169.09 - risk $102.75 = $66.34 if all refund


## 3. Before/after replies (EN) — 4 examples

Rule: **apologize → cite file:line → promise 2h fix (reship/refund choice)**. Never invent order numbers.

In [7]:
before = "Where is my order? No tracking."
after = ("Hi Alice! Sorry about the delay — I pulled your note "
         "([instagram_dm.txt:4]) and am checking tracking now. "
         "Update in 2h; reship/refund your choice.")
print(f"BEFORE ({len(before)}ch): {before}")
print(f"AFTER  ({len(after)}ch): {after}")

BEFORE (31ch): Where is my order? No tracking.
AFTER  (146ch): Hi Alice! Sorry about the delay — I pulled your note ([instagram_dm.txt:4]) and am checking tracking now. Update in 2h; reship/refund your choice.


### #1 Late delivery (DM)

BEFORE ignores; AFTER names the pain, cites the line, and gives a 2h fix with choice. Reply-time target: <2h.

In [8]:
print('BEFORE: Where is my order??')
print('AFTER: Hi! Sorry it is late — tracking checked, update in 2h; reship/refund your choice. [instagram_dm.txt:4]')
print('why better: apology + citation + choice + deadline')

BEFORE: Where is my order??
AFTER: Hi! Sorry it is late — tracking checked, update in 2h; reship/refund your choice. [instagram_dm.txt:4]
why better: apology + citation + choice + deadline


### #2 Damaged arrival (review)

BEFORE is defensive; AFTER thanks, cites, fixes packing, offers replacement/refund publicly (future buyers read it).

In [9]:
print('BEFORE: Our mugs rarely break.')
print('AFTER: Thanks for flagging ([reviews.csv:3]) — sorry! Packing fixed; DM for free replacement/refund.')
print('why better: public accountability, not argument')

BEFORE: Our mugs rarely break.
AFTER: Thanks for flagging ([reviews.csv:3]) — sorry! Packing fixed; DM for free replacement/refund.
why better: public accountability, not argument


### #3 Refund demand (DM)

BEFORE stalls; AFTER processes today, cites, and moves to DM for address/choice. Speed beats wording here.

In [10]:
print('BEFORE: Please see our policy page.')
print('AFTER: Processing your refund today ([instagram_dm.txt:1]) — sorry! Confirm reship vs refund in DM?')
print('why better: same-day action + question that resolves')

BEFORE: Please see our policy page.
AFTER: Processing your refund today ([instagram_dm.txt:1]) — sorry! Confirm reship vs refund in DM?
why better: same-day action + question that resolves


### #4 Happy message (DM → review ask)

BEFORE wastes love; AFTER thanks + asks for review/UGC with citation. Happy msgs fund tomorrow's sales.

In [11]:
print('BEFORE: Thanks!')
print('AFTER: So glad you love it! ([instagram_dm.txt:2]) Would you post a quick review/photo? Thank you!')
print('why better: converts love into listing proof')

BEFORE: Thanks!
AFTER: So glad you love it! ([instagram_dm.txt:2]) Would you post a quick review/photo? Thank you!
why better: converts love into listing proof


### i18n pointer — same shape, 5 locales

`src/omni_one/i18n/` ships `en/es/zh/fr/de` (`drafts.build_reply`, `build_review_reply`). Same citation + choice structure, localized apology/fix phrasing. Use `list_locales()` → `['en','es','zh','fr','de']`; `detect_locale(text)` routes CJK/accents.

In [12]:
locales = ['en', 'es', 'zh', 'fr', 'de']
print(f"locales: {', '.join(locales)}")
print('en: Sorry about the delay — update in 2h; reship/refund your choice.')
print('es: Perdón por el retraso — actualización en 2h; reenvío/reembolso a elegir.')
print('rule: build_reply(channel, locale, customer, text, citation, shop)')

locales: en, es, zh, fr, de
en: Sorry about the delay — update in 2h; reship/refund your choice.
es: Perdón por el retraso — actualización en 2h; reenvío/reembolso a elegir.
rule: build_reply(channel, locale, customer, text, citation, shop)


## 4. Decision tree — refund / reship / escalate

```
refund-demand + chargeback/fraud word? ──YES──▶ ESCALATE (pull evidence, do not reship blind)
│ NO
late + tracking stalled >4d? ──YES──▶ RESHIP (or refund — customer choice)
│ NO
damaged/broken with photo? ──YES──▶ RESHIP + packing fix note
│ NO
question / address change <1h? ──YES──▶ FIX + confirm
else ──▶ REPLY + watch 24h
```

In [13]:
def decide(kind, stalled_days=0, photo=False, fraud=False):
    if kind=='refund-demand' and fraud:
        return 'ESCALATE'
    if kind=='refund-demand' or (kind=='late' and stalled_days>4):
        return 'RES HIP'.replace(' ', '')
    if kind=='damaged' and photo:
        return 'RESHIP'
    if kind=='question':
        return 'FIX'
    return 'REPLY+WATCH'

for case in [('refund-demand',0,False,True),('late',5,False,False),('damaged',0,True,False),('question',0,False,False)]:
    print(f"{case} -> {decide(*case)}")

('refund-demand', 0, False, True) -> ESCALATE
('late', 5, False, False) -> RESHIP
('damaged', 0, True, False) -> RESHIP
('question', 0, False, False) -> FIX


### Escalation checklist

- [ ] P0 replies <2h (refund-demand first)
- [ ] Every draft cites `file:line`
- [ ] Offer reship/refund choice (not policy link)
- [ ] Chargeback/fraud → escalate with evidence, no blind reship
- [ ] Happy → review/UGC ask same day

In [14]:
p0, sla = 3, 2
print(f"today: {p0} P0 x 20min = {p0*20}min (SLA {sla}h -> {'PASS' if p0*20<=sla*60 else 'FAIL'})")
print('money protected $102.75; profit shielded $169.09')

today: 3 P0 x 20min = 60min (SLA 2h -> PASS)
money protected $102.75; profit shielded $169.09
